[**RAG**](https://drive.google.com/file/d/1MOfoqYJhiYjdektaRPtIGdBWEJ9OFPbv/view?usp=drive_link)

![描述](https://r2.epsiotapi.com/Summer-School-2026/RAG_SummerSchool.jpg)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#STEP 1 — Install Required Libraries

In [2]:
%pip install -U pandas numpy faiss-cpu FlagEmbedding openai


# STEP 2 — Import Libraries


In [3]:
import pandas as pd
import numpy as np
import faiss
import os

from FlagEmbedding import FlagModel
from openai import OpenAI

# STEP 3 — Load Dataset




# [Here](https://r2.epsiotapi.com/Summer-School-2026/rag_tutorial_corpus_100_docs.xlsx) is Dataset.


In [5]:
# file_path = "/content/drive/MyDrive/Summer School 2026/Yelp-Restaurant-Reviews.csv"
file_path = "/content/drive/MyDrive/Summer School 2026/materials/rag_tutorial_corpus_100_docs.xlsx"
df = pd.read_excel(file_path)

# Display first rows
print(df.head())

   doc_id               title  \
0       1  Company BlueWave 1   
1       2  Company BlueWave 2   
2       3  Company BlueWave 3   
3       4  Company BlueWave 4   
4       5  Company BlueWave 5   

                                             content  
0  Company: BlueWave 1\nFounded: 2011\nHeadquarte...  
1  Company: BlueWave 2\nFounded: 2012\nHeadquarte...  
2  Company: BlueWave 3\nFounded: 2013\nHeadquarte...  
3  Company: BlueWave 4\nFounded: 2014\nHeadquarte...  
4  Company: BlueWave 5\nFounded: 2015\nHeadquarte...  


In [ ]:
'''# You also can upload the file from local.
from google.colab import files

_ = files.upload()
file_name = ""

with open(file_name, "r") as f:
    example = f.read()
print(example)

# STEP 4 — Create Corpus


In [6]:
# Select review text column
# Change column name if necessary

corpus = df["title"].astype(str).tolist()

print("Number of documents:", len(corpus))


Number of documents: 100


In [7]:
corpus = df.apply(
    lambda row: f"""
Column1: {row['doc_id']}
Column2: {row['title']}
Column3: {row['content']}
""".strip(),
    axis=1
).tolist()

In [8]:
corpus

['Column1: 1\nColumn2: Company BlueWave 1\nColumn3: Company: BlueWave 1\nFounded: 2011\nHeadquarters: City1\nCEO: Alex Carter 1\nProducts:\n- Product1A\n- Product1B',
 'Column1: 2\nColumn2: Company BlueWave 2\nColumn3: Company: BlueWave 2\nFounded: 2012\nHeadquarters: City2\nCEO: Alex Carter 2\nProducts:\n- Product2A\n- Product2B',
 'Column1: 3\nColumn2: Company BlueWave 3\nColumn3: Company: BlueWave 3\nFounded: 2013\nHeadquarters: City3\nCEO: Alex Carter 3\nProducts:\n- Product3A\n- Product3B',
 'Column1: 4\nColumn2: Company BlueWave 4\nColumn3: Company: BlueWave 4\nFounded: 2014\nHeadquarters: City4\nCEO: Alex Carter 4\nProducts:\n- Product4A\n- Product4B',
 'Column1: 5\nColumn2: Company BlueWave 5\nColumn3: Company: BlueWave 5\nFounded: 2015\nHeadquarters: City5\nCEO: Alex Carter 5\nProducts:\n- Product5A\n- Product5B',
 'Column1: 6\nColumn2: Company BlueWave 6\nColumn3: Company: BlueWave 6\nFounded: 2016\nHeadquarters: City6\nCEO: Alex Carter 6\nProducts:\n- Product6A\n- Product6B'

# STEP 5 — Load BGE Embedding Model
Beijing Academy of Artificial Intelligence (BAAI)

In [9]:
model = FlagModel(
    'BAAI/bge-base-en-v1.5',
    query_instruction_for_retrieval="Represent this sentence for searching relevant passages:",
    use_fp16=True
)

print("BGE model loaded successfully.")

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE model loaded successfully.


# STEP 6 — Generate Embeddings


In [10]:
embeddings = model.encode(
    corpus,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (100, 768)


# STEP 7 — Build FAISS Index




In [11]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Number of indexed documents:", index.ntotal)

Number of indexed documents: 100


# STEP 8 — User Query

In [12]:
user_input = "I want to learn more about Head Quarter and CEO and the building of Company BlueWave 1."


# STEP 9 — Encode Query


In [13]:
q_embedding = model.encode_queries(
    [user_input],
    convert_to_numpy=True
)

# STEP 10 — Retrieve Top Documents


In [14]:
top_k = 7

D, I = index.search(q_embedding, top_k)

retrieved_docs = np.array(corpus)[I[0]]

print("\nRetrieved Reviews:\n")

for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}:\n{doc}\n")


Retrieved Reviews:

Document 1:
Column1: 1
Column2: Company BlueWave 1
Column3: Company: BlueWave 1
Founded: 2011
Headquarters: City1
CEO: Alex Carter 1
Products:
- Product1A
- Product1B

Document 2:
Column1: 34
Column2: Company BlueWave 34
Column3: Company: BlueWave 34
Founded: 2014
Headquarters: City34
CEO: Alex Carter 34
Products:
- Product34A
- Product34B

Document 3:
Column1: 39
Column2: Company BlueWave 39
Column3: Company: BlueWave 39
Founded: 2019
Headquarters: City39
CEO: Alex Carter 39
Products:
- Product39A
- Product39B

Document 4:
Column1: 31
Column2: Company BlueWave 31
Column3: Company: BlueWave 31
Founded: 2011
Headquarters: City31
CEO: Alex Carter 31
Products:
- Product31A
- Product31B

Document 5:
Column1: 35
Column2: Company BlueWave 35
Column3: Company: BlueWave 35
Founded: 2015
Headquarters: City35
CEO: Alex Carter 35
Products:
- Product35A
- Product35B

Document 6:
Column1: 16
Column2: Company BlueWave 16
Column3: Company: BlueWave 16
Founded: 2016
Headquarters: 

# STEP 11 — Create RAG Context


In [15]:
context = "\n\n".join(retrieved_docs)
context

'Column1: 1\nColumn2: Company BlueWave 1\nColumn3: Company: BlueWave 1\nFounded: 2011\nHeadquarters: City1\nCEO: Alex Carter 1\nProducts:\n- Product1A\n- Product1B\n\nColumn1: 34\nColumn2: Company BlueWave 34\nColumn3: Company: BlueWave 34\nFounded: 2014\nHeadquarters: City34\nCEO: Alex Carter 34\nProducts:\n- Product34A\n- Product34B\n\nColumn1: 39\nColumn2: Company BlueWave 39\nColumn3: Company: BlueWave 39\nFounded: 2019\nHeadquarters: City39\nCEO: Alex Carter 39\nProducts:\n- Product39A\n- Product39B\n\nColumn1: 31\nColumn2: Company BlueWave 31\nColumn3: Company: BlueWave 31\nFounded: 2011\nHeadquarters: City31\nCEO: Alex Carter 31\nProducts:\n- Product31A\n- Product31B\n\nColumn1: 35\nColumn2: Company BlueWave 35\nColumn3: Company: BlueWave 35\nFounded: 2015\nHeadquarters: City35\nCEO: Alex Carter 35\nProducts:\n- Product35A\n- Product35B\n\nColumn1: 16\nColumn2: Company BlueWave 16\nColumn3: Company: BlueWave 16\nFounded: 2016\nHeadquarters: City16\nCEO: Alex Carter 16\nProducts:

# STEP 12 — Create Prompt


# WITHOUT RAG

In [18]:
prompt = f"""
You are a helpful assistant.





User Request:
{user_input}

Provide:
1. Exact match recommendation
2. full details of the requested query
If you do not have the information say that you do not know.
"""

# WITH RAG

---



In [29]:
prompt = f"""
You are a helpful recommendation assistant.

Use ONLY the retrieved reviews below
to answer the user's request.

Retrieved Reviews:
{context}

User Request:
{user_input}

Provide:
1. Exact match recommendation
2. full details of the requested query

"""

# STEP 13 — OpenRouter API Key


In [30]:
'''###DIRECT INJECTION
os.environ["OPENROUTER_KEY"] = ""

In [31]:
# Replace with your own API key
from google.colab import userdata
os.environ["OPENROUTER_KEY"] = userdata.get('OPENROUTER_KEY')

# STEP 14 — Initialize OpenRouter Client


In [32]:
client = OpenAI(
    api_key=os.environ["OPENROUTER_KEY"],
    base_url="https://openrouter.ai/api/v1"
)



# STEP 15 — Generate Final Response


In [33]:
response = client.chat.completions.create(
    model="openai/gpt-3.5-turbo-16k",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.7,
    max_tokens=300
)


# STEP 16 — Display Final Answer


In [34]:

print("\n================ RAG RESPONSE ================\n")

print(response.choices[0].message.content)


================ RAG RESPONSE ================

1. Exact match recommendation:
Company BlueWave 1

2. Full details of the requested query:
- Company Name: BlueWave 1
- Founded: 2011
- Headquarters: City1
- CEO: Alex Carter 1
